# 07. PSI Monitoring

Goal: measure population stability of the frozen scorecard (`models/scorecard_pipeline.json`).

- Expected (baseline) population: training set
- Observed populations: (a) validation set, expected to show PSI near zero (sanity baseline), (b) `application_test.csv`, a genuinely unseen population used as a proxy for incoming applicants
- Note: the dataset has no calendar dates, so (b) is not true temporal drift. It is the best available proxy for a new population.
- PSI is computed per scorecard characteristic and on the final score, always using the frozen training bins (never recalculated).

In [1]:
# Imports and paths
import json
import numpy as np
import pandas as pd
import sys
sys.path.append('../src')
from scorecard_utils import score_client

DATA_RAW = '../data/raw/'
DATA_PROC = '../data/processed/'

with open('../models/scorecard_pipeline.json') as f:
    pipeline = json.load(f)

print(len(pipeline['variables']), 'variables in the frozen pipeline')

17 variables in the frozen pipeline


In [2]:
# Load application_test and replicate, line by line, the treatment applied to application_train in 02
app_test = pd.read_csv(DATA_RAW + 'application_test.csv')
print(app_test.shape)  # 48744 rows expected (my prior knowledge of the dataset, please confirm)

app_test['DAYS_EMPLOYED_ANOM'] = (app_test['DAYS_EMPLOYED'] == 365243).astype('int8')
app_test['DAYS_EMPLOYED'] = app_test['DAYS_EMPLOYED'].replace(365243, np.nan)

app_test['AGE_YEARS'] = (-app_test['DAYS_BIRTH'] / 365).round(1)
app_test['YEARS_EMPLOYED'] = (-app_test['DAYS_EMPLOYED'] / 365).round(1)
app_test['YEARS_REGISTRATION'] = (-app_test['DAYS_REGISTRATION'] / 365).round(1)
app_test['YEARS_ID_PUBLISH'] = (-app_test['DAYS_ID_PUBLISH'] / 365).round(1)
app_test['YEARS_LAST_PHONE_CHANGE'] = (-app_test['DAYS_LAST_PHONE_CHANGE'] / 365).round(1)
app_test = app_test.copy()

(48744, 121)


C:\Users\vitor\AppData\Local\Temp\ipykernel_10752\2020182197.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_test['DAYS_EMPLOYED_ANOM'] = (app_test['DAYS_EMPLOYED'] == 365243).astype('int8')
C:\Users\vitor\AppData\Local\Temp\ipykernel_10752\2020182197.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_test['AGE_YEARS'] = (-app_test['DAYS_BIRTH'] / 365).round(1)
C:\Users\vitor\AppData\Local\Temp\ipykernel_10752\2020182197.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling

In [ ]:
# ASSUMPTION TO VERIFY: bureau_agg exists on disk. Adjust the filename to what you actually saved.
bureau_agg = pd.read_csv(DATA_PROC + 'bureau_agg.csv')

test_ids = set(app_test['SK_ID_CURR'])
agg_ids = set(bureau_agg['SK_ID_CURR'])
print('Test clients with bureau history:', len(test_ids & agg_ids), 'of', len(test_ids))

test_bureau = app_test.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(test_bureau.shape)